In [1]:
# ============================================================
# Notebook 20: Feature-observability and timestamp-lag audit
# Single-cell standalone version
#
# Purpose:
# 1. Audit whether final modeling features satisfy market observability rules.
# 2. Focus especially on U.S. variables:
#    S&P 500, NASDAQ, SOXX, VIX, U.S. 10-year Treasury yield.
# 3. Report:
#    - source timestamp convention
#    - effective date used
#    - lag applied
#    - decision timestamp convention
#    - number of violations
#    - number of rows changed after proposed correction
#
# Main outputs:
# - table_S41a_feature_observability_audit_summary.csv
# - table_S41b_feature_column_lag_audit.csv
# - table_S41c_empirical_source_match_audit.csv
# - table_S41d_rows_changed_after_correction.csv
# - table_S41e_source_panel_inventory.csv
# - table_S41f_observability_status_for_manuscript.csv
# - AURORA_TWETF_features_with_labels_observability_corrected.parquet/csv
# - NOTEBOOK20_validation_report.json
# - NOTEBOOK20_file_manifest_SHA256.csv
#
# Important:
# This notebook can verify definite timestamp problems when feature names or
# empirical source matches identify same-day external-market usage. If feature
# names do not encode source/lag information and no raw source panel is available,
# affected columns are reported as "ambiguous" rather than falsely declared safe.
#
# Research audit only. Not investment advice.
# ============================================================

from __future__ import annotations

import json
import math
import re
import hashlib
import warnings
from pathlib import Path
from datetime import datetime, timezone

warnings.filterwarnings("ignore")

try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception:
    print("Google Drive mount skipped or failed.")

import numpy as np
import pandas as pd

# ============================================================
# 1. Paths and configuration
# ============================================================

PROJECT_CODE = "AURORA_TWETF"
PUBLICATION_ROOT = Path("/content/drive/MyDrive/AURORA_TWETF")

DATA_ROOT = PUBLICATION_ROOT / "data"
PANEL_DIR = DATA_ROOT / "panels"
MODELING_DIR = DATA_ROOT / "modeling"

OUTPUT_ROOT = PUBLICATION_ROOT / "outputs" / PROJECT_CODE
TABLE_DIR = OUTPUT_ROOT / "tables"
REPORT_DIR = OUTPUT_ROOT / "reports"

RUN_TIMESTAMP = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")

RUN_ROOT = OUTPUT_ROOT / "feature_observability_audit" / f"run_{RUN_ID}"
TABLE_RUN_DIR = RUN_ROOT / "tables"
REPORT_RUN_DIR = RUN_ROOT / "reports"
DIAG_DIR = RUN_ROOT / "diagnostics"
CORRECTED_DIR = RUN_ROOT / "corrected_feature_matrix"

for d in [
    RUN_ROOT,
    TABLE_RUN_DIR,
    REPORT_RUN_DIR,
    DIAG_DIR,
    CORRECTED_DIR,
    TABLE_DIR,
    REPORT_DIR,
]:
    d.mkdir(parents=True, exist_ok=True)

# Main modeling matrix. The parquet file is preferred because Notebook 07 uses it.
MODEL_DATA_CANDIDATES = [
    MODELING_DIR / "AURORA_TWETF_features_with_labels.parquet",
    MODELING_DIR / "AURORA_TWETF_features_with_labels.csv",
]

# Conservative settings.
# If True, ambiguous external-market features are shifted by the required lag in the corrected copy.
# Default False because ambiguous columns should trigger metadata review rather than silent correction.
SHIFT_AMBIGUOUS_EXTERNAL_FEATURES = False

# If True, use conservative one-Taiwan-trading-day lag for all non-Taiwan regional/FX variables.
# U.S. variables always require at least one Taiwan-trading-day lag.
CONSERVATIVE_LAG_FOR_REGIONAL_AND_FX = True

# Correlation threshold for detecting an empirical exact/same-day source match.
EMPIRICAL_MATCH_CORR_THRESHOLD = 0.999
MIN_EMPIRICAL_MATCH_OBS = 80

# Date convention for the forecast/allocation decision.
DECISION_TIMESTAMP_CONVENTION = (
    "Taiwan after-close forecast/allocation timestamp; same-calendar-date U.S. market "
    "close is not assumed observable before the Taiwan decision."
)

print("=" * 100)
print("AURORA-TWETF Notebook 20")
print("Feature-observability and timestamp-lag audit")
print("=" * 100)
print("RUN_ID:", RUN_ID)
print("RUN_ROOT:", RUN_ROOT)
print("=" * 100)

# ============================================================
# 2. Helper functions
# ============================================================

def save_json(path, obj):
    Path(path).write_text(
        json.dumps(obj, indent=2, ensure_ascii=False, default=str),
        encoding="utf-8",
    )

def sha256_file(path, chunk_size=1024 * 1024):
    path = Path(path)
    h = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(chunk_size), b""):
            h.update(chunk)
    return h.hexdigest()

def make_file_manifest(root):
    root = Path(root)
    rows = []
    for p in sorted(root.rglob("*")):
        if p.is_file():
            stat = p.stat()
            rows.append({
                "path": p.relative_to(root).as_posix(),
                "size_bytes": int(stat.st_size),
                "modified_utc": datetime.fromtimestamp(
                    stat.st_mtime,
                    timezone.utc,
                ).strftime("%Y-%m-%dT%H:%M:%SZ"),
                "sha256": sha256_file(p),
            })
    return pd.DataFrame(rows)

def write_table(df, filename_stem):
    local_csv = TABLE_RUN_DIR / f"{filename_stem}.csv"
    global_csv = TABLE_DIR / f"{filename_stem}_{RUN_ID}.csv"
    df.to_csv(local_csv, index=False)
    df.to_csv(global_csv, index=False)
    print("Saved:", local_csv)
    print("Saved:", global_csv)
    return local_csv, global_csv

def normalize_name(x):
    return re.sub(r"[^a-zA-Z0-9]+", "", str(x)).lower()

def safe_name(x):
    return (
        str(x)
        .replace("/", "_")
        .replace("\\", "_")
        .replace(":", "_")
        .replace(" ", "_")
        .replace(".", "_")
        .replace("%", "pct")
        .replace("-", "_")
        .replace("+", "plus")
        .replace("=", "_")
        .replace("(", "")
        .replace(")", "")
    )

def looks_like_date_series(s):
    parsed = pd.to_datetime(s, errors="coerce")
    if not isinstance(parsed, pd.Series):
        parsed = pd.Series(parsed)
    if parsed.notna().mean() < 0.50:
        return False, parsed
    years = parsed.dt.year
    if years.between(1990, 2035).mean() < 0.50:
        return False, parsed
    if parsed.nunique(dropna=True) < min(10, max(2, len(parsed) // 10)):
        return False, parsed
    return True, parsed

def set_datetime_index_flex(df):
    df = df.copy()

    if isinstance(df.index, pd.DatetimeIndex):
        years = pd.Series(df.index.year)
        if years.between(1990, 2035).mean() > 0.50:
            df.index = pd.to_datetime(df.index)
            df.index.name = "date"
            df = df[~df.index.isna()]
            return df.sort_index()

    preferred_cols = [
        "date", "Date", "DATE",
        "datetime", "Datetime", "DATETIME",
        "timestamp", "Timestamp", "TIMESTAMP",
        "time", "Time", "TIME",
        "Unnamed: 0", "index", "Index",
    ]

    candidate_cols = [c for c in preferred_cols if c in df.columns] + [
        c for c in df.columns if c not in preferred_cols
    ]

    for c in candidate_cols:
        try:
            ok, parsed = looks_like_date_series(df[c])
            if ok:
                df = df.drop(columns=[c])
                df.index = pd.to_datetime(parsed)
                df.index.name = "date"
                df = df[~df.index.isna()]
                return df.sort_index()
        except Exception:
            continue

    idx_series = pd.Series(df.index)
    ok, parsed_idx = looks_like_date_series(idx_series)
    if ok:
        df.index = pd.to_datetime(parsed_idx.values)
        df.index.name = "date"
        df = df[~df.index.isna()]
        return df.sort_index()

    return df

def read_table_auto_flex(path):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"Missing file: {path}")

    if path.suffix.lower() == ".parquet":
        df = pd.read_parquet(path)
    elif path.suffix.lower() == ".csv":
        df = pd.read_csv(path, low_memory=False)
    else:
        raise ValueError(f"Unsupported file type: {path}")

    return set_datetime_index_flex(df)

def find_first_existing(paths):
    for p in paths:
        p = Path(p)
        if p.exists():
            return p
    return None

def is_target_or_metadata_col(col):
    c = str(col).lower()
    metadata_tokens = [
        "target",
        "label",
        "regime_fixed",
        "forward",
        "fwd",
        "split",
        "fold",
        "date",
        "timestamp",
        "run_id",
        "model",
        "proba_class",
        "pred",
        "prediction",
    ]
    return any(tok in c for tok in metadata_tokens)

def parse_explicit_lag_from_feature_name(col):
    """
    Infer lag from feature name when possible.
    Returns integer lag in Taiwan trading-day units if explicitly encoded.
    0 means same-calendar-day / lag0.
    Negative values mean lead/future-like naming.
    None means not inferable from the name.
    """
    raw = str(col).lower()
    compact = normalize_name(raw)

    # Explicit future/lead patterns.
    future_patterns = [
        r"lead[_\-]?(\d+)",
        r"future[_\-]?(\d+)",
        r"tplus[_\-]?(\d+)",
        r"t\+(\d+)",
    ]
    for pat in future_patterns:
        m = re.search(pat, raw)
        if m:
            return -int(m.group(1))
        m = re.search(pat, compact)
        if m:
            return -int(m.group(1))

    if any(tok in compact for tok in ["sameday", "lag0", "shift0", "t0"]):
        return 0

    if any(tok in compact for tok in ["prev", "previous", "yesterday"]):
        return 1

    patterns = [
        r"(?:^|[_\-])lag[_\-]?(\d+)(?:$|[_\-])",
        r"lag(\d+)",
        r"(?:^|[_\-])shift[_\-]?(\d+)(?:$|[_\-])",
        r"shift(\d+)",
        r"tminus[_\-]?(\d+)",
        r"tm[_\-]?(\d+)",
        r"t_(\d+)",
        r"t-(\d+)",
    ]

    for pat in patterns:
        m = re.search(pat, raw)
        if m:
            return int(m.group(1))
        m = re.search(pat, compact)
        if m:
            return int(m.group(1))

    return None

def effective_date_description(lag):
    if lag is None:
        return "not inferable from feature name"
    if lag < 0:
        return f"future date t+{abs(lag)} trading day(s)"
    if lag == 0:
        return "same calendar/trading date t"
    return f"t-{lag} Taiwan trading day(s)"

def zscore_series(s):
    s = pd.Series(s).astype(float)
    sd = s.std(skipna=True)
    if not np.isfinite(sd) or sd == 0:
        return s * np.nan
    return (s - s.mean(skipna=True)) / sd

# ============================================================
# 3. Observability policy table
# ============================================================

def required_lag_for_group(default_lag):
    return int(default_lag)

VARIABLE_SPECS = [
    {
        "source_variable": "TAIEX",
        "variable_group": "Taiwan market index",
        "market_region": "Taiwan",
        "tokens": ["taiex", "twii", "taiwanweighted", "taiwancapitalization"],
        "required_lag_tw_days": 0,
        "same_calendar_observable": "Yes, if decision occurs after Taiwan close",
        "source_timestamp_convention": "Taiwan market close on date t",
    },
    {
        "source_variable": "0050",
        "variable_group": "Taiwan ETF prices/returns",
        "market_region": "Taiwan",
        "tokens": ["0050", "0050tw"],
        "required_lag_tw_days": 0,
        "same_calendar_observable": "Yes, if decision occurs after Taiwan close",
        "source_timestamp_convention": "Taiwan ETF close on date t",
    },
    {
        "source_variable": "006208",
        "variable_group": "Taiwan ETF prices/returns",
        "market_region": "Taiwan",
        "tokens": ["006208", "006208tw"],
        "required_lag_tw_days": 0,
        "same_calendar_observable": "Yes, if decision occurs after Taiwan close",
        "source_timestamp_convention": "Taiwan ETF close on date t",
    },
    {
        "source_variable": "00692",
        "variable_group": "Taiwan ETF prices/returns",
        "market_region": "Taiwan",
        "tokens": ["00692", "00692tw"],
        "required_lag_tw_days": 0,
        "same_calendar_observable": "Yes, if decision occurs after Taiwan close",
        "source_timestamp_convention": "Taiwan ETF close on date t",
    },
    {
        "source_variable": "00881",
        "variable_group": "Taiwan ETF prices/returns",
        "market_region": "Taiwan",
        "tokens": ["00881", "00881tw"],
        "required_lag_tw_days": 0,
        "same_calendar_observable": "Yes, if decision occurs after Taiwan close",
        "source_timestamp_convention": "Taiwan ETF close on date t",
    },
    {
        "source_variable": "S&P 500",
        "variable_group": "U.S. broad market",
        "market_region": "United States",
        "tokens": ["sp500", "snp500", "sandp500", "gspc", "spx", "sp500index"],
        "required_lag_tw_days": 1,
        "same_calendar_observable": "No for Taiwan same-calendar-date decisions",
        "source_timestamp_convention": "U.S. market close on calendar date t; after Taiwan date-t close",
    },
    {
        "source_variable": "NASDAQ",
        "variable_group": "U.S. technology market",
        "market_region": "United States",
        "tokens": ["nasdaq", "ixic", "ndx", "nasdaqcomposite"],
        "required_lag_tw_days": 1,
        "same_calendar_observable": "No for Taiwan same-calendar-date decisions",
        "source_timestamp_convention": "U.S. market close on calendar date t; after Taiwan date-t close",
    },
    {
        "source_variable": "SOXX",
        "variable_group": "U.S. semiconductor proxy",
        "market_region": "United States",
        "tokens": ["soxx", "icesemiconductor"],
        "required_lag_tw_days": 1,
        "same_calendar_observable": "No for Taiwan same-calendar-date decisions",
        "source_timestamp_convention": "U.S. ETF close on calendar date t; after Taiwan date-t close",
    },
    {
        "source_variable": "VIX",
        "variable_group": "U.S. volatility proxy",
        "market_region": "United States",
        "tokens": ["vix", "vx"],
        "required_lag_tw_days": 1,
        "same_calendar_observable": "No for Taiwan same-calendar-date decisions",
        "source_timestamp_convention": "U.S. market close on calendar date t; after Taiwan date-t close",
    },
    {
        "source_variable": "U.S. 10-year Treasury yield",
        "variable_group": "U.S. interest-rate proxy",
        "market_region": "United States",
        "tokens": ["us10y", "ust10y", "10y", "tnx", "treasury10", "us10", "us10year", "yield10"],
        "required_lag_tw_days": 1,
        "same_calendar_observable": "No for Taiwan same-calendar-date decisions",
        "source_timestamp_convention": "U.S. rate observation on calendar date t; not assumed observable before Taiwan date-t decision",
    },
    {
        "source_variable": "Nikkei 225",
        "variable_group": "Regional Asian equity markets",
        "market_region": "Japan",
        "tokens": ["nikkei", "n225", "nk225", "nikkei225"],
        "required_lag_tw_days": 1 if CONSERVATIVE_LAG_FOR_REGIONAL_AND_FX else 0,
        "same_calendar_observable": "Depends on decision timestamp",
        "source_timestamp_convention": "Japan market close; conservative audit uses latest prior observable value",
    },
    {
        "source_variable": "Hang Seng",
        "variable_group": "Regional Asian equity markets",
        "market_region": "Hong Kong",
        "tokens": ["hangseng", "hsi", "hstech"],
        "required_lag_tw_days": 1 if CONSERVATIVE_LAG_FOR_REGIONAL_AND_FX else 0,
        "same_calendar_observable": "Depends on decision timestamp",
        "source_timestamp_convention": "Hong Kong market close; conservative audit uses latest prior observable value",
    },
    {
        "source_variable": "KOSPI",
        "variable_group": "Regional Asian equity markets",
        "market_region": "Korea",
        "tokens": ["kospi", "ks11"],
        "required_lag_tw_days": 1 if CONSERVATIVE_LAG_FOR_REGIONAL_AND_FX else 0,
        "same_calendar_observable": "Depends on decision timestamp",
        "source_timestamp_convention": "Korea market close; conservative audit uses latest prior observable value",
    },
    {
        "source_variable": "USD/TWD",
        "variable_group": "Foreign-exchange proxy",
        "market_region": "FX / Taiwan data source",
        "tokens": ["usdtwd", "twdusd", "usdntd", "ntdusd", "twd"],
        "required_lag_tw_days": 1 if CONSERVATIVE_LAG_FOR_REGIONAL_AND_FX else 0,
        "same_calendar_observable": "Depends on data timestamp",
        "source_timestamp_convention": "FX/source-specific timestamp; conservative audit uses latest prior observable value",
    },
]

# Sort tokens by length so specific tokens match first.
for spec in VARIABLE_SPECS:
    spec["tokens"] = sorted(list(set([normalize_name(t) for t in spec["tokens"]])), key=len, reverse=True)

policy_df = pd.DataFrame([
    {
        "source_variable": s["source_variable"],
        "variable_group": s["variable_group"],
        "market_region": s["market_region"],
        "source_timestamp_convention": s["source_timestamp_convention"],
        "decision_timestamp_convention": DECISION_TIMESTAMP_CONVENTION,
        "same_calendar_observable_before_taiwan_decision": s["same_calendar_observable"],
        "minimum_required_lag_tw_trading_days": s["required_lag_tw_days"],
    }
    for s in VARIABLE_SPECS
])

write_table(policy_df, "table_S41_policy_observability_rules")

# ============================================================
# 4. Load final modeling matrix
# ============================================================

print("\n" + "=" * 100)
print("Loading final modeling matrix")
print("=" * 100)

model_data_path = find_first_existing(MODEL_DATA_CANDIDATES)

if model_data_path is None:
    # Broad fallback search.
    candidates = list(MODELING_DIR.rglob("AURORA_TWETF_features_with_labels*.parquet")) + \
                 list(MODELING_DIR.rglob("AURORA_TWETF_features_with_labels*.csv")) + \
                 list(DATA_ROOT.rglob("AURORA_TWETF_features_with_labels*.parquet")) + \
                 list(DATA_ROOT.rglob("AURORA_TWETF_features_with_labels*.csv"))
    if not candidates:
        raise FileNotFoundError(
            "Could not find AURORA_TWETF_features_with_labels.parquet/csv. "
            "Please place the final modeling matrix in /data/modeling."
        )
    model_data_path = sorted(candidates, key=lambda p: p.stat().st_mtime, reverse=True)[0]

model_df = read_table_auto_flex(model_data_path)

if not isinstance(model_df.index, pd.DatetimeIndex):
    raise ValueError("Modeling matrix does not have a valid DatetimeIndex after flexible parsing.")

model_df = model_df[~model_df.index.duplicated(keep="last")].sort_index()

numeric_cols = [
    c for c in model_df.columns
    if pd.api.types.is_numeric_dtype(model_df[c])
]

feature_cols = [
    c for c in numeric_cols
    if not is_target_or_metadata_col(c)
]

target_cols = [
    c for c in model_df.columns
    if is_target_or_metadata_col(c)
]

print("Modeling matrix:", model_data_path)
print("Shape:", model_df.shape)
print("Date range:", model_df.index.min().date(), "to", model_df.index.max().date())
print("Numeric columns:", len(numeric_cols))
print("Candidate feature columns:", len(feature_cols))
print("Target/metadata columns:", len(target_cols))

# ============================================================
# 5. Optional feature-metadata search
# ============================================================

print("\n" + "=" * 100)
print("Searching for feature metadata / feature manifest files")
print("=" * 100)

metadata_patterns = [
    "*feature*metadata*.csv",
    "*feature*manifest*.csv",
    "*feature*index*.csv",
    "*features*metadata*.csv",
    "*features*manifest*.csv",
    "*features*index*.csv",
]

metadata_candidates = []
for root in [MODELING_DIR, DATA_ROOT, OUTPUT_ROOT]:
    root = Path(root)
    if root.exists():
        for pat in metadata_patterns:
            metadata_candidates.extend(list(root.rglob(pat)))

metadata_candidates = sorted(
    list(set([p for p in metadata_candidates if p.is_file()])),
    key=lambda p: p.stat().st_mtime,
    reverse=True,
)

metadata_inventory_rows = []
metadata_df = None
metadata_path = None

for p in metadata_candidates:
    try:
        tmp = pd.read_csv(p, low_memory=False)
        metadata_inventory_rows.append({
            "path": str(p),
            "n_rows": len(tmp),
            "columns": ",".join(map(str, tmp.columns[:20])),
            "selected": False,
        })

        cols_norm = {normalize_name(c): c for c in tmp.columns}
        has_feature_col = any(k in cols_norm for k in ["feature", "featurename", "column", "columnname"])
        if metadata_df is None and has_feature_col and len(tmp) >= 10:
            metadata_df = tmp.copy()
            metadata_path = p
            metadata_inventory_rows[-1]["selected"] = True

    except Exception as e:
        metadata_inventory_rows.append({
            "path": str(p),
            "n_rows": np.nan,
            "columns": "",
            "selected": False,
            "error": repr(e)[:120],
        })

metadata_inventory_df = pd.DataFrame(metadata_inventory_rows)
write_table(metadata_inventory_df, "table_S41_metadata_file_inventory")

if metadata_path is not None:
    print("Selected feature metadata:", metadata_path)
else:
    print("No usable feature metadata found; falling back to feature-name and empirical source-match audit.")

# ============================================================
# 6. Classify feature columns by source variable and parse lags
# ============================================================

print("\n" + "=" * 100)
print("Classifying feature columns and parsing explicit lags")
print("=" * 100)

def classify_feature_column(col):
    cn = normalize_name(col)
    matches = []
    for spec in VARIABLE_SPECS:
        for tok in spec["tokens"]:
            if tok and tok in cn:
                matches.append((len(tok), spec))
                break
    if not matches:
        return None
    matches = sorted(matches, key=lambda x: x[0], reverse=True)
    return matches[0][1]

def get_metadata_for_feature(col):
    if metadata_df is None:
        return {}
    cols = list(metadata_df.columns)
    norm_map = {normalize_name(c): c for c in cols}

    feature_col = None
    for key in ["feature", "featurename", "column", "columnname"]:
        if key in norm_map:
            feature_col = norm_map[key]
            break

    if feature_col is None:
        return {}

    rows = metadata_df[metadata_df[feature_col].astype(str) == str(col)]
    if rows.empty:
        rows = metadata_df[metadata_df[feature_col].astype(str).map(normalize_name) == normalize_name(col)]

    if rows.empty:
        return {}

    row = rows.iloc[0].to_dict()

    out = {}
    for k, v in row.items():
        nk = normalize_name(k)
        if nk in ["source", "sourcevariable", "basevariable", "ticker", "symbol"]:
            out["metadata_source_variable"] = v
        elif nk in ["lag", "lagdays", "lagtradingdays", "effective_lag", "featurelag"]:
            try:
                out["metadata_lag"] = int(float(v))
            except Exception:
                out["metadata_lag_raw"] = v
        elif nk in ["effectivedate", "effectivedateused", "sourceeffectivedate"]:
            out["metadata_effective_date"] = v
        elif nk in ["sourcetimestamp", "source_time", "sourcedatetime"]:
            out["metadata_source_timestamp"] = v

    return out

column_rows = []

for col in feature_cols:
    spec = classify_feature_column(col)
    explicit_lag = parse_explicit_lag_from_feature_name(col)
    md = get_metadata_for_feature(col)

    if spec is None and "metadata_source_variable" in md:
        # Try metadata-based source classification.
        md_source_norm = normalize_name(md["metadata_source_variable"])
        for candidate_spec in VARIABLE_SPECS:
            if any(tok in md_source_norm for tok in candidate_spec["tokens"]):
                spec = candidate_spec
                break

    if "metadata_lag" in md:
        inferred_lag = md["metadata_lag"]
        lag_source = "metadata"
    else:
        inferred_lag = explicit_lag
        lag_source = "feature_name" if explicit_lag is not None else "not_inferable"

    if spec is None:
        required_lag = np.nan
        source_variable = "unclassified"
        variable_group = "unclassified"
        market_region = "unclassified"
        source_timestamp = "not classified"
        same_calendar = "not classified"
        explicit_violation = False
        audit_scope = "unclassified"
        effective_date = effective_date_description(inferred_lag)
    else:
        required_lag = int(spec["required_lag_tw_days"])
        source_variable = spec["source_variable"]
        variable_group = spec["variable_group"]
        market_region = spec["market_region"]
        source_timestamp = spec["source_timestamp_convention"]
        same_calendar = spec["same_calendar_observable"]
        explicit_violation = inferred_lag is not None and inferred_lag < required_lag
        audit_scope = "in_scope"
        effective_date = effective_date_description(inferred_lag)

    column_rows.append({
        "feature_column": col,
        "audit_scope": audit_scope,
        "source_variable": source_variable,
        "variable_group": variable_group,
        "market_region": market_region,
        "source_timestamp_convention": source_timestamp,
        "decision_timestamp_convention": DECISION_TIMESTAMP_CONVENTION,
        "same_calendar_observable_before_taiwan_decision": same_calendar,
        "minimum_required_lag_tw_trading_days": required_lag,
        "inferred_lag_tw_trading_days": inferred_lag,
        "lag_inference_source": lag_source,
        "effective_date_used": effective_date,
        "explicit_lag_violation": bool(explicit_violation),
        "metadata_source_variable": md.get("metadata_source_variable", ""),
        "metadata_lag": md.get("metadata_lag", np.nan),
        "metadata_effective_date": md.get("metadata_effective_date", ""),
        "metadata_source_timestamp": md.get("metadata_source_timestamp", ""),
    })

column_audit_df = pd.DataFrame(column_rows)

# Add initial status.
column_audit_df["requires_external_lag"] = (
    pd.to_numeric(column_audit_df["minimum_required_lag_tw_trading_days"], errors="coerce").fillna(0) > 0
)

column_audit_df["lag_ambiguous"] = (
    (column_audit_df["audit_scope"] == "in_scope")
    & (column_audit_df["requires_external_lag"])
    & (column_audit_df["inferred_lag_tw_trading_days"].isna())
)

write_table(column_audit_df, "table_S41b_feature_column_lag_audit_initial")

print("Classified feature columns:", int((column_audit_df["audit_scope"] == "in_scope").sum()))
print("External-lag-required columns:", int(column_audit_df["requires_external_lag"].sum()))
print("Explicit lag violations:", int(column_audit_df["explicit_lag_violation"].sum()))
print("Ambiguous lag columns:", int(column_audit_df["lag_ambiguous"].sum()))

# ============================================================
# 7. Search raw/source panels for empirical source matching
# ============================================================

print("\n" + "=" * 100)
print("Searching raw/source panels for empirical observability audit")
print("=" * 100)

def source_file_candidates():
    patterns = [
        "*.parquet",
        "*.csv",
    ]
    candidates = []
    for root in [PANEL_DIR, DATA_ROOT]:
        root = Path(root)
        if not root.exists():
            continue
        for pat in patterns:
            candidates.extend(list(root.rglob(pat)))

    # Exclude final modeling matrix unless no source panels exist.
    filtered = []
    for p in candidates:
        path_str = p.as_posix().lower()
        if "features_with_labels" in path_str:
            continue
        if "modeling" in path_str and "panel" not in path_str:
            continue
        filtered.append(p)

    filtered = sorted(
        list(set([p for p in filtered if p.is_file()])),
        key=lambda p: (
            0 if "panel" in p.as_posix().lower() else 1,
            0 if "close" in p.as_posix().lower() or "price" in p.as_posix().lower() else 1,
            len(p.as_posix()),
        ),
    )

    return filtered

def find_source_series_for_variable(spec, max_files=200):
    candidates = source_file_candidates()
    candidates = candidates[:max_files]

    rows = []
    best = None

    for p in candidates:
        try:
            df = read_table_auto_flex(p)
            if not isinstance(df.index, pd.DatetimeIndex):
                continue
            if len(df) < 100:
                continue

            for c in df.columns:
                cn = normalize_name(c)
                token_match = any(tok in cn for tok in spec["tokens"])
                if not token_match:
                    continue

                s = pd.to_numeric(df[c], errors="coerce")
                coverage = float(s.notna().mean())
                if coverage < 0.50:
                    continue

                score = coverage
                if "close" in normalize_name(c):
                    score += 0.20
                if "adj" in normalize_name(c):
                    score += 0.15
                if "return" in normalize_name(c) or "ret" in normalize_name(c):
                    score += 0.10
                if spec["source_variable"].replace("/", "").lower() in normalize_name(c):
                    score += 0.25

                row = {
                    "source_variable": spec["source_variable"],
                    "path": str(p),
                    "column": str(c),
                    "n_rows": int(len(df)),
                    "date_start": str(df.index.min().date()),
                    "date_end": str(df.index.max().date()),
                    "coverage": coverage,
                    "score": score,
                }
                rows.append(row)

                if best is None or score > best["score"]:
                    best = row | {"series": s.copy(), "index": df.index.copy()}

        except Exception:
            continue

    return best, pd.DataFrame(rows)

source_inventory_frames = []
source_series_by_variable = {}

for spec in VARIABLE_SPECS:
    best, inv = find_source_series_for_variable(spec)
    if not inv.empty:
        source_inventory_frames.append(inv)

    if best is not None:
        s = pd.Series(best["series"].values, index=pd.to_datetime(best["index"]), name=best["column"])
        s = s[~s.index.duplicated(keep="last")].sort_index()
        source_series_by_variable[spec["source_variable"]] = {
            "series": s,
            "path": best["path"],
            "column": best["column"],
            "score": best["score"],
        }
        print(f"Source found for {spec['source_variable']}: {best['path']} | {best['column']}")
    else:
        print(f"No source series found for {spec['source_variable']}")

source_inventory_df = (
    pd.concat(source_inventory_frames, ignore_index=True)
    if source_inventory_frames
    else pd.DataFrame(columns=["source_variable", "path", "column", "n_rows", "date_start", "date_end", "coverage", "score"])
)

write_table(source_inventory_df, "table_S41e_source_panel_inventory")

# ============================================================
# 8. Empirical source-match audit
# ============================================================

print("\n" + "=" * 100)
print("Running empirical source-match audit")
print("=" * 100)

def build_source_transforms(s):
    """
    Candidate transforms for empirical matching.
    If the source series is already a return series, level_z can still match.
    If it is a price level, pct/log/diff transforms can match engineered returns.
    """
    s = pd.Series(s).astype(float).sort_index()
    out = {}

    out["level"] = s
    out["level_z"] = zscore_series(s)
    out["diff_1"] = s.diff(1)
    out["diff_1_z"] = zscore_series(s.diff(1))

    # Return transforms. They will be mostly NaN if source contains negative values.
    with np.errstate(divide="ignore", invalid="ignore"):
        out["pct_change_1"] = s.pct_change(1)
        out["pct_change_5"] = s.pct_change(5)
        out["pct_change_20"] = s.pct_change(20)
        positive_s = s.where(s > 0)
        out["logret_1"] = np.log(positive_s).diff(1)
        out["logret_5"] = np.log(positive_s).diff(5)
        out["logret_20"] = np.log(positive_s).diff(20)

    out["rolling_mean_5_pct"] = out["pct_change_1"].rolling(5).mean()
    out["rolling_mean_20_pct"] = out["pct_change_1"].rolling(20).mean()
    out["rolling_std_5_pct"] = out["pct_change_1"].rolling(5).std()
    out["rolling_std_20_pct"] = out["pct_change_1"].rolling(20).std()

    # Remove transforms with insufficient variation.
    clean = {}
    for k, v in out.items():
        v = pd.Series(v).replace([np.inf, -np.inf], np.nan)
        if v.notna().sum() < MIN_EMPIRICAL_MATCH_OBS:
            continue
        if v.std(skipna=True) == 0 or not np.isfinite(v.std(skipna=True)):
            continue
        clean[k] = v

    return clean

def compute_corr(a, b):
    x = pd.concat([pd.Series(a), pd.Series(b)], axis=1).dropna()
    if len(x) < MIN_EMPIRICAL_MATCH_OBS:
        return np.nan, len(x)
    if x.iloc[:, 0].std() == 0 or x.iloc[:, 1].std() == 0:
        return np.nan, len(x)
    return float(x.iloc[:, 0].corr(x.iloc[:, 1])), len(x)

def empirical_match_for_feature(feature_series, source_series, candidate_shifts=(-2, -1, 0, 1, 2, 3, 4, 5)):
    transforms = build_source_transforms(source_series)
    best = {
        "best_abs_corr": np.nan,
        "best_corr": np.nan,
        "best_transform": "",
        "best_lag_tw_trading_days": np.nan,
        "best_n_obs": 0,
    }

    f = pd.Series(feature_series).astype(float).sort_index()
    f = f.replace([np.inf, -np.inf], np.nan)

    if f.notna().sum() < MIN_EMPIRICAL_MATCH_OBS or f.std(skipna=True) == 0:
        return best

    for transform_name, source_transform in transforms.items():
        source_transform = pd.Series(source_transform).sort_index()

        for lag in candidate_shifts:
            # Candidate effective source date for feature date t is t-lag.
            # To align source value at t-lag to feature date t, shift source forward by lag.
            candidate = source_transform.shift(lag)
            candidate = candidate.reindex(f.index)
            corr, nobs = compute_corr(f, candidate)

            if not np.isfinite(corr):
                continue

            abs_corr = abs(corr)

            if not np.isfinite(best["best_abs_corr"]) or abs_corr > best["best_abs_corr"]:
                best = {
                    "best_abs_corr": float(abs_corr),
                    "best_corr": float(corr),
                    "best_transform": transform_name,
                    "best_lag_tw_trading_days": int(lag),
                    "best_n_obs": int(nobs),
                }

    return best

empirical_rows = []

# Audit only columns that map to a source variable with a required lag or important external sources.
audit_candidate_df = column_audit_df[
    (column_audit_df["audit_scope"] == "in_scope")
    & (column_audit_df["source_variable"].isin(source_series_by_variable.keys()))
].copy()

for _, row in audit_candidate_df.iterrows():
    feature_col = row["feature_column"]
    source_variable = row["source_variable"]
    required_lag = int(row["minimum_required_lag_tw_trading_days"])
    source_obj = source_series_by_variable[source_variable]

    match = empirical_match_for_feature(
        feature_series=model_df[feature_col],
        source_series=source_obj["series"],
    )

    exact_like = (
        np.isfinite(match["best_abs_corr"])
        and match["best_abs_corr"] >= EMPIRICAL_MATCH_CORR_THRESHOLD
    )

    empirical_violation = (
        exact_like
        and int(match["best_lag_tw_trading_days"]) < required_lag
    )

    empirical_rows.append({
        "feature_column": feature_col,
        "source_variable": source_variable,
        "required_lag_tw_trading_days": required_lag,
        "source_panel_path": source_obj["path"],
        "source_panel_column": source_obj["column"],
        "best_transform": match["best_transform"],
        "best_lag_tw_trading_days": match["best_lag_tw_trading_days"],
        "best_corr": match["best_corr"],
        "best_abs_corr": match["best_abs_corr"],
        "best_n_obs": match["best_n_obs"],
        "exact_or_near_exact_empirical_match": bool(exact_like),
        "empirical_timestamp_violation": bool(empirical_violation),
        "empirical_effective_date_used": effective_date_description(
            int(match["best_lag_tw_trading_days"])
            if np.isfinite(match["best_lag_tw_trading_days"])
            else None
        ),
    })

empirical_audit_df = pd.DataFrame(empirical_rows)

if empirical_audit_df.empty:
    empirical_audit_df = pd.DataFrame(columns=[
        "feature_column",
        "source_variable",
        "required_lag_tw_trading_days",
        "source_panel_path",
        "source_panel_column",
        "best_transform",
        "best_lag_tw_trading_days",
        "best_corr",
        "best_abs_corr",
        "best_n_obs",
        "exact_or_near_exact_empirical_match",
        "empirical_timestamp_violation",
        "empirical_effective_date_used",
    ])

write_table(empirical_audit_df, "table_S41c_empirical_source_match_audit")

print("Empirical matches checked:", len(empirical_audit_df))
print("Empirical timestamp violations:", int(empirical_audit_df["empirical_timestamp_violation"].sum()) if len(empirical_audit_df) else 0)

# ============================================================
# 9. Combine explicit-lag and empirical-match audits
# ============================================================

print("\n" + "=" * 100)
print("Combining explicit and empirical audit results")
print("=" * 100)

combined_df = column_audit_df.copy()

emp_cols = [
    "feature_column",
    "best_transform",
    "best_lag_tw_trading_days",
    "best_corr",
    "best_abs_corr",
    "best_n_obs",
    "exact_or_near_exact_empirical_match",
    "empirical_timestamp_violation",
    "empirical_effective_date_used",
    "source_panel_path",
    "source_panel_column",
]

combined_df = combined_df.merge(
    empirical_audit_df[[c for c in emp_cols if c in empirical_audit_df.columns]],
    on="feature_column",
    how="left",
)

combined_df["exact_or_near_exact_empirical_match"] = combined_df["exact_or_near_exact_empirical_match"].fillna(False)
combined_df["empirical_timestamp_violation"] = combined_df["empirical_timestamp_violation"].fillna(False)

def final_violation(row):
    return bool(row.get("explicit_lag_violation", False)) or bool(row.get("empirical_timestamp_violation", False))

def recommended_shift_amount(row):
    required_lag = row.get("minimum_required_lag_tw_trading_days", np.nan)
    if not np.isfinite(required_lag):
        return 0

    required_lag = int(required_lag)

    if bool(row.get("empirical_timestamp_violation", False)) and np.isfinite(row.get("best_lag_tw_trading_days", np.nan)):
        detected_lag = int(row["best_lag_tw_trading_days"])
        return max(required_lag - detected_lag, 0)

    if bool(row.get("explicit_lag_violation", False)) and pd.notna(row.get("inferred_lag_tw_trading_days", np.nan)):
        detected_lag = int(row["inferred_lag_tw_trading_days"])
        return max(required_lag - detected_lag, 0)

    if (
        SHIFT_AMBIGUOUS_EXTERNAL_FEATURES
        and bool(row.get("lag_ambiguous", False))
        and bool(row.get("requires_external_lag", False))
    ):
        return required_lag

    return 0

def audit_status(row):
    if row["audit_scope"] != "in_scope":
        return "not_applicable_unclassified"
    if final_violation(row):
        return "violation_detected"
    if row["requires_external_lag"] and row["lag_ambiguous"] and not row["exact_or_near_exact_empirical_match"]:
        return "ambiguous_requires_feature_metadata"
    if row["requires_external_lag"] and row["exact_or_near_exact_empirical_match"]:
        return "verified_by_empirical_match"
    if row["requires_external_lag"] and pd.notna(row["inferred_lag_tw_trading_days"]):
        return "verified_by_explicit_lag_name_or_metadata"
    if not row["requires_external_lag"]:
        return "no_external_lag_required"
    return "review_required"

combined_df["final_timestamp_violation"] = combined_df.apply(final_violation, axis=1)
combined_df["recommended_shift_tw_trading_days"] = combined_df.apply(recommended_shift_amount, axis=1)
combined_df["audit_status"] = combined_df.apply(audit_status, axis=1)

# If empirical match exists, prefer empirical effective-date description.
combined_df["effective_date_used_final"] = combined_df["effective_date_used"]
mask_emp_effective = combined_df["empirical_effective_date_used"].notna() & (combined_df["empirical_effective_date_used"] != "")
combined_df.loc[mask_emp_effective, "effective_date_used_final"] = combined_df.loc[mask_emp_effective, "empirical_effective_date_used"]

write_table(combined_df, "table_S41b_feature_column_lag_audit")

violating_cols = combined_df.loc[combined_df["final_timestamp_violation"], "feature_column"].tolist()
ambiguous_cols = combined_df.loc[
    (combined_df["audit_scope"] == "in_scope")
    & (combined_df["requires_external_lag"])
    & (combined_df["audit_status"] == "ambiguous_requires_feature_metadata"),
    "feature_column"
].tolist()

print("Final definite timestamp-violation feature columns:", len(violating_cols))
print("Ambiguous external-lag-required feature columns:", len(ambiguous_cols))

# ============================================================
# 10. Proposed correction and rows changed after correction
# ============================================================

print("\n" + "=" * 100)
print("Constructing proposed observability-corrected feature matrix")
print("=" * 100)

corrected_df = model_df.copy()

correction_rows = []

# Correction is applied only to definite violations by default.
# If SHIFT_AMBIGUOUS_EXTERNAL_FEATURES=True, ambiguous required-lag columns are also shifted.
correction_candidate_df = combined_df[
    (combined_df["final_timestamp_violation"])
    | (
        SHIFT_AMBIGUOUS_EXTERNAL_FEATURES
        & (combined_df["audit_status"] == "ambiguous_requires_feature_metadata")
    )
].copy()

for _, row in correction_candidate_df.iterrows():
    col = row["feature_column"]
    if col not in corrected_df.columns:
        continue

    shift_amount = int(row["recommended_shift_tw_trading_days"])
    if shift_amount <= 0:
        continue

    old = corrected_df[col].copy()
    new = old.shift(shift_amount)
    corrected_df[col] = new

    changed_mask = ~(
        old.eq(new)
        | (old.isna() & new.isna())
    )

    correction_rows.append({
        "feature_column": col,
        "source_variable": row["source_variable"],
        "audit_status": row["audit_status"],
        "minimum_required_lag_tw_trading_days": row["minimum_required_lag_tw_trading_days"],
        "detected_best_lag_tw_trading_days": row.get("best_lag_tw_trading_days", np.nan),
        "inferred_lag_tw_trading_days": row.get("inferred_lag_tw_trading_days", np.nan),
        "recommended_shift_tw_trading_days": shift_amount,
        "non_missing_before": int(old.notna().sum()),
        "non_missing_after": int(new.notna().sum()),
        "n_rows_changed": int(changed_mask.sum()),
        "first_changed_date": str(corrected_df.index[changed_mask].min().date()) if changed_mask.any() else "",
        "last_changed_date": str(corrected_df.index[changed_mask].max().date()) if changed_mask.any() else "",
    })

correction_df = pd.DataFrame(correction_rows)

if correction_df.empty:
    correction_df = pd.DataFrame(columns=[
        "feature_column",
        "source_variable",
        "audit_status",
        "minimum_required_lag_tw_trading_days",
        "detected_best_lag_tw_trading_days",
        "inferred_lag_tw_trading_days",
        "recommended_shift_tw_trading_days",
        "non_missing_before",
        "non_missing_after",
        "n_rows_changed",
        "first_changed_date",
        "last_changed_date",
    ])

# Date-level rows changed.
if correction_rows:
    changed_any = pd.Series(False, index=model_df.index)
    for row in correction_rows:
        col = row["feature_column"]
        old = model_df[col]
        new = corrected_df[col]
        changed_any = changed_any | (~(old.eq(new) | (old.isna() & new.isna())))
    n_dates_changed = int(changed_any.sum())
    first_date_changed = str(changed_any.index[changed_any].min().date()) if changed_any.any() else ""
    last_date_changed = str(changed_any.index[changed_any].max().date()) if changed_any.any() else ""
else:
    changed_any = pd.Series(False, index=model_df.index)
    n_dates_changed = 0
    first_date_changed = ""
    last_date_changed = ""

rows_changed_summary = pd.DataFrame([{
    "n_modeling_rows": int(len(model_df)),
    "n_features_with_definite_timestamp_violation": int(len(violating_cols)),
    "n_ambiguous_external_lag_required_features": int(len(ambiguous_cols)),
    "shift_ambiguous_external_features": bool(SHIFT_AMBIGUOUS_EXTERNAL_FEATURES),
    "n_features_shifted_in_corrected_copy": int(len(correction_df)),
    "n_dates_changed_after_correction": int(n_dates_changed),
    "first_changed_date": first_date_changed,
    "last_changed_date": last_date_changed,
    "correction_policy": (
        "Definite timestamp violations shifted by the minimum additional lag required. "
        "Ambiguous external features are shifted only if SHIFT_AMBIGUOUS_EXTERNAL_FEATURES=True."
    ),
}])

write_table(correction_df, "table_S41d_feature_corrections")
write_table(rows_changed_summary, "table_S41d_rows_changed_after_correction")

# Save corrected matrix. If no violations, this is identical to the original and serves as a checksum artifact.
corrected_parquet = CORRECTED_DIR / "AURORA_TWETF_features_with_labels_observability_corrected.parquet"
corrected_csv = CORRECTED_DIR / "AURORA_TWETF_features_with_labels_observability_corrected.csv"

corrected_df.to_parquet(corrected_parquet)
corrected_df.to_csv(corrected_csv)

print("Saved corrected feature matrix:")
print(corrected_parquet)
print(corrected_csv)

# ============================================================
# 11. Summary table for manuscript
# ============================================================

print("\n" + "=" * 100)
print("Creating manuscript-ready observability status table")
print("=" * 100)

summary_rows = []

for spec in VARIABLE_SPECS:
    source_variable = spec["source_variable"]
    sub = combined_df[combined_df["source_variable"] == source_variable].copy()

    feature_count = int(len(sub))
    external_required = int(spec["required_lag_tw_days"]) > 0

    definite_viols = int(sub["final_timestamp_violation"].sum()) if feature_count else 0
    ambiguous = int((sub["audit_status"] == "ambiguous_requires_feature_metadata").sum()) if feature_count else 0
    verified_explicit = int((sub["audit_status"] == "verified_by_explicit_lag_name_or_metadata").sum()) if feature_count else 0
    verified_empirical = int((sub["audit_status"] == "verified_by_empirical_match").sum()) if feature_count else 0
    no_lag_req = int((sub["audit_status"] == "no_external_lag_required").sum()) if feature_count else 0

    corr_sub = correction_df[correction_df["source_variable"] == source_variable].copy()
    rows_changed = int(corr_sub["n_rows_changed"].sum()) if not corr_sub.empty else 0

    if feature_count == 0:
        status = "No feature columns detected by source-name audit"
        interpretation = "No source-specific feature columns were identified from feature names or metadata."
    elif definite_viols > 0:
        status = "Violation detected"
        interpretation = "At least one feature appears to use information earlier than the required observability lag."
    elif ambiguous > 0:
        status = "No definite violation, but metadata review required"
        interpretation = (
            "No exact same-day empirical match or explicit violation was detected, but some external-source "
            "features do not encode lag/source metadata clearly enough for full verification."
        )
    else:
        status = "Zero detected timestamp violations"
        interpretation = "All detected source-specific columns satisfy the required lag by explicit metadata/name or empirical match."

    summary_rows.append({
        "source_variable": source_variable,
        "variable_group": spec["variable_group"],
        "market_region": spec["market_region"],
        "source_timestamp_convention": spec["source_timestamp_convention"],
        "decision_timestamp_convention": DECISION_TIMESTAMP_CONVENTION,
        "minimum_required_lag_tw_trading_days": spec["required_lag_tw_days"],
        "feature_columns_detected": feature_count,
        "verified_by_explicit_lag_or_metadata": verified_explicit,
        "verified_by_empirical_match": verified_empirical,
        "no_external_lag_required": no_lag_req,
        "ambiguous_columns_requiring_metadata_review": ambiguous,
        "definite_timestamp_violations": definite_viols,
        "rows_changed_after_proposed_correction": rows_changed,
        "audit_status": status,
        "interpretation": interpretation,
    })

summary_df = pd.DataFrame(summary_rows)

write_table(summary_df, "table_S41a_feature_observability_audit_summary")

# Overall status for manuscript wording.
total_definite_violations = int(summary_df["definite_timestamp_violations"].sum())
total_ambiguous = int(summary_df["ambiguous_columns_requiring_metadata_review"].sum())
total_detected_features = int(summary_df["feature_columns_detected"].sum())
total_rows_changed = int(rows_changed_summary["n_dates_changed_after_correction"].iloc[0])

if total_definite_violations == 0 and total_ambiguous == 0:
    overall_status = "The audit found zero detected timestamp violations."
    manuscript_sentence = (
        "The automated feature-observability audit found zero detected timestamp violations across "
        "classified market-state feature columns."
    )
elif total_definite_violations == 0 and total_ambiguous > 0:
    overall_status = "No definite timestamp violations detected, but ambiguous columns require feature metadata review."
    manuscript_sentence = (
        "The automated feature-observability audit found no definite timestamp violations, but "
        f"{total_ambiguous} external-source feature columns lacked sufficient source/lag metadata for full verification."
    )
else:
    overall_status = "Timestamp violations detected; rerun affected pipeline before claiming complete leakage control."
    manuscript_sentence = (
        f"The automated feature-observability audit detected {total_definite_violations} feature columns "
        f"with timestamp-lag violations; a corrected feature matrix changed {total_rows_changed} dates. "
        "The affected forecasting/allocation analyses should be rerun before claiming complete leakage control."
    )

status_df = pd.DataFrame([{
    "run_id": RUN_ID,
    "modeling_matrix": str(model_data_path),
    "n_modeling_rows": int(len(model_df)),
    "n_candidate_feature_columns": int(len(feature_cols)),
    "n_classified_market_state_feature_columns": int(total_detected_features),
    "n_definite_timestamp_violation_columns": int(total_definite_violations),
    "n_ambiguous_external_lag_required_columns": int(total_ambiguous),
    "n_dates_changed_after_proposed_correction": int(total_rows_changed),
    "overall_status": overall_status,
    "manuscript_sentence": manuscript_sentence,
}])

write_table(status_df, "table_S41f_observability_status_for_manuscript")

print("\nOverall observability audit status:")
print(status_df.to_string(index=False))

# ============================================================
# 12. Optional detailed violation examples
# ============================================================

violation_details_df = combined_df[
    (combined_df["final_timestamp_violation"])
    | (combined_df["audit_status"] == "ambiguous_requires_feature_metadata")
].copy()

write_table(violation_details_df, "table_S41g_violation_and_ambiguous_feature_details")

# Short examples of changed rows, if any.
if n_dates_changed > 0:
    examples = []
    changed_dates = changed_any.index[changed_any].tolist()[:20]
    changed_feature_cols = correction_df["feature_column"].tolist()[:20]

    for dt in changed_dates:
        row = {"date": dt}
        for col in changed_feature_cols:
            row[f"{col}_original"] = model_df.loc[dt, col] if col in model_df.columns else np.nan
            row[f"{col}_corrected"] = corrected_df.loc[dt, col] if col in corrected_df.columns else np.nan
        examples.append(row)

    change_examples_df = pd.DataFrame(examples)
else:
    change_examples_df = pd.DataFrame(columns=["date"])

write_table(change_examples_df, "table_S41h_correction_changed_row_examples")

# ============================================================
# 13. Validation report and manifest
# ============================================================

validation_report = {
    "project_code": PROJECT_CODE,
    "notebook": "20_AURORA_feature_observability_timestamp_lag_audit",
    "run_timestamp_utc": RUN_TIMESTAMP,
    "run_id": RUN_ID,
    "purpose": (
        "Automated audit of whether final modeling-matrix feature columns satisfy market-source "
        "observability and lag conventions, especially for non-Taiwan market variables."
    ),
    "input_paths": {
        "modeling_matrix": str(model_data_path),
        "feature_metadata_path": str(metadata_path) if metadata_path is not None else None,
    },
    "configuration": {
        "decision_timestamp_convention": DECISION_TIMESTAMP_CONVENTION,
        "shift_ambiguous_external_features": bool(SHIFT_AMBIGUOUS_EXTERNAL_FEATURES),
        "conservative_lag_for_regional_and_fx": bool(CONSERVATIVE_LAG_FOR_REGIONAL_AND_FX),
        "empirical_match_corr_threshold": EMPIRICAL_MATCH_CORR_THRESHOLD,
        "minimum_empirical_match_observations": MIN_EMPIRICAL_MATCH_OBS,
    },
    "modeling_matrix": {
        "n_rows": int(len(model_df)),
        "n_columns": int(model_df.shape[1]),
        "n_candidate_feature_columns": int(len(feature_cols)),
        "date_start": str(model_df.index.min().date()),
        "date_end": str(model_df.index.max().date()),
    },
    "audit_results": {
        "n_classified_market_state_feature_columns": int(total_detected_features),
        "n_definite_timestamp_violation_columns": int(total_definite_violations),
        "n_ambiguous_external_lag_required_columns": int(total_ambiguous),
        "n_dates_changed_after_proposed_correction": int(total_rows_changed),
        "overall_status": overall_status,
        "manuscript_sentence": manuscript_sentence,
    },
    "source_series_found": {
        k: {
            "path": v["path"],
            "column": v["column"],
            "score": v["score"],
        }
        for k, v in source_series_by_variable.items()
    },
    "output_paths": {
        "run_root": str(RUN_ROOT),
        "tables": str(TABLE_RUN_DIR),
        "diagnostics": str(DIAG_DIR),
        "corrected_feature_matrix_parquet": str(corrected_parquet),
        "corrected_feature_matrix_csv": str(corrected_csv),
        "table_S41a_summary": str(TABLE_RUN_DIR / "table_S41a_feature_observability_audit_summary.csv"),
        "table_S41b_column_audit": str(TABLE_RUN_DIR / "table_S41b_feature_column_lag_audit.csv"),
        "table_S41c_empirical_audit": str(TABLE_RUN_DIR / "table_S41c_empirical_source_match_audit.csv"),
        "table_S41d_rows_changed": str(TABLE_RUN_DIR / "table_S41d_rows_changed_after_correction.csv"),
        "table_S41f_status": str(TABLE_RUN_DIR / "table_S41f_observability_status_for_manuscript.csv"),
    },
    "limitations": (
        "The audit verifies definite timestamp-lag problems when feature names, feature metadata, or "
        "empirical raw-source matching identify effective lags. Columns without source/lag metadata and "
        "without exact empirical source matches are reported as ambiguous; they require feature-construction "
        "metadata or Notebook 01/02 inspection for full proof."
    ),
}

validation_report_path = REPORT_RUN_DIR / "NOTEBOOK20_validation_report.json"
validation_report_global_path = REPORT_DIR / f"NOTEBOOK20_validation_report_{RUN_ID}.json"

save_json(validation_report_path, validation_report)
save_json(validation_report_global_path, validation_report)

manifest_df = make_file_manifest(RUN_ROOT)

manifest_path = REPORT_RUN_DIR / "NOTEBOOK20_file_manifest_SHA256.csv"
manifest_global_path = REPORT_DIR / f"NOTEBOOK20_file_manifest_SHA256_{RUN_ID}.csv"

manifest_df.to_csv(manifest_path, index=False)
manifest_df.to_csv(manifest_global_path, index=False)

# ============================================================
# 14. Final console summary
# ============================================================

print("\n" + "=" * 100)
print("AURORA-TWETF NOTEBOOK 20 COMPLETE")
print("=" * 100)
print("Run ID:", RUN_ID)
print("Run root:", RUN_ROOT)
print("Modeling matrix:", model_data_path)
print("Candidate feature columns:", len(feature_cols))
print("Classified market-state feature columns:", total_detected_features)
print("Definite timestamp-violation columns:", total_definite_violations)
print("Ambiguous external-lag-required columns:", total_ambiguous)
print("Rows/dates changed after proposed correction:", total_rows_changed)
print("Overall status:", overall_status)
print("Manuscript sentence:", manuscript_sentence)
print("\nKey output tables:")
print("S41a summary:", TABLE_RUN_DIR / "table_S41a_feature_observability_audit_summary.csv")
print("S41b column audit:", TABLE_RUN_DIR / "table_S41b_feature_column_lag_audit.csv")
print("S41c empirical audit:", TABLE_RUN_DIR / "table_S41c_empirical_source_match_audit.csv")
print("S41d rows changed:", TABLE_RUN_DIR / "table_S41d_rows_changed_after_correction.csv")
print("S41f manuscript status:", TABLE_RUN_DIR / "table_S41f_observability_status_for_manuscript.csv")
print("Corrected matrix:", corrected_parquet)
print("Validation report:", validation_report_path)
print("Manifest:", manifest_path)
print("=" * 100)

print("\nTable S41f preview:")
print(status_df.to_string(index=False))

print("\nTable S41a preview:")
print(summary_df.to_string(index=False))

if total_definite_violations > 0 or total_ambiguous > 0:
    print("\nViolation/ambiguous feature preview:")
    print(violation_details_df.head(50).to_string(index=False))
else:
    print("\nNo definite violations or ambiguous external-lag-required features detected.")

print("\nRows-changed summary:")
print(rows_changed_summary.to_string(index=False))

Mounted at /content/drive
AURORA-TWETF Notebook 20
Feature-observability and timestamp-lag audit
RUN_ID: 20260718_154326
RUN_ROOT: /content/drive/MyDrive/AURORA_TWETF/outputs/AURORA_TWETF/feature_observability_audit/run_20260718_154326
Saved: /content/drive/MyDrive/AURORA_TWETF/outputs/AURORA_TWETF/feature_observability_audit/run_20260718_154326/tables/table_S41_policy_observability_rules.csv
Saved: /content/drive/MyDrive/AURORA_TWETF/outputs/AURORA_TWETF/tables/table_S41_policy_observability_rules_20260718_154326.csv

Loading final modeling matrix
Modeling matrix: /content/drive/MyDrive/AURORA_TWETF/data/modeling/AURORA_TWETF_features_with_labels.parquet
Shape: (1262, 417)
Date range: 2021-01-06 to 2026-03-25
Numeric columns: 417
Candidate feature columns: 415
Target/metadata columns: 2

Searching for feature metadata / feature manifest files
Saved: /content/drive/MyDrive/AURORA_TWETF/outputs/AURORA_TWETF/feature_observability_audit/run_20260718_154326/tables/table_S41_metadata_file_i